# 03 同じ費用でベースラインとモデルを比較する

## Goal

PITの合成信号を使い、zero、train mean、ridge、tree、buy-and-holdを同じ期間・同じ費用で比較します。計算は共通APIの compare_model_strategies に委ねます。

## Setup

- 入力: DEMO:ALPHAのlockbox開始前46足。02と同じ時点別snapshotを各判断時刻に保存
- 信号: 3期モメンタムと3期ボラティリティ。1期先ラベルは利用可能時刻を明示
- 検証: 12期学習、3期テスト、1期embargo、3期step。lockbox以降は評価しない
- 配分: 正の予測ならlong、その他はcash。目標ウェイトはエンジンが1期だけ遅らせる
- 費用: 手数料3bpsとスリッページ2bps。初回建玉、売却、反転に同じ式を適用
- 価格: USD / raw。FX、実約定、実口座は含めません

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import plotly.graph_objects as go
from market_research.research.signals import build_pit_signal_table
from market_research.research.strategy_compare import compare_model_strategies
from market_research.services import build_demo_run
from market_research.storage import CacheKey, ResearchStore

run = build_demo_run()
lockbox_start = run.prices.index[46].to_pydatetime()
selected_bars = tuple(
    bar for bar in run.bars
    if bar.instrument.instrument_id == "DEMO:ALPHA"
    and bar.available_at < lockbox_start
)

## Steps

### 1. 保存済み版から信号を再構築する

02に依存せず単独で実行できます。合成snapshotは一時領域にだけ置きます。

In [ ]:
with TemporaryDirectory(prefix="market-research-notebook-") as directory:
    with ResearchStore(Path(directory)) as store:
        snapshot_ids = []
        for bar in selected_bars:
            key = CacheKey(
                bar.provider, "prices", bar.instrument.instrument_id,
                bar.interval, bar.instrument.currency, bar.adjustment,
            )
            raw = f"synthetic:{bar.bar_end.isoformat()}:{bar.close:.10f}".encode()
            saved = store.save(
                key, raw, observed_at=bar.observed_at, prices=(bar,),
            )
            snapshot_ids.append(saved.snapshot_id)
        signals = build_pit_signal_table(
            store,
            instrument_id="DEMO:ALPHA",
            snapshot_ids=tuple(snapshot_ids),
            decision_times=tuple(bar.available_at for bar in selected_bars),
            expected_bar_ends=tuple(bar.bar_end for bar in selected_bars),
            currency="USD",
            adjustment="raw",
            momentum_window=3,
            volatility_window=3,
            horizon=1,
        )
print(f"run_id={run.run_id}, snapshots={len(snapshot_ids)}, lockbox={lockbox_start.isoformat()}")

### 2. OOS予測を1期遅れの配分へ変換する

訓練時点の未来ラベルがテスト開始時に公表済みかを共通APIが検査します。zeroは常にcashの下限、buy-and-holdは同期間の参照です。

In [ ]:
comparison = compare_model_strategies(
    signals,
    train_size=12,
    test_size=3,
    embargo=1,
    step=3,
    lockbox_start=lockbox_start,
    commission_bps=3,
    slippage_bps=2,
)
scores = pd.DataFrame(
    {
        "strategy": name,
        "equity_end": float(result.equity.iloc[-1]),
        "net_return_pct": 100 * (float(result.equity.iloc[-1]) - 1),
        "turnover_sum": float(result.turnover.sum()),
        "cost_sum_pct_points": 100 * float(result.costs.sum()),
    }
    for name, result in comparison.backtests.items()
).sort_values("equity_end", ascending=False)
print(f"OOS decisions={len(comparison.evaluation.table)}, evaluation_as_of={comparison.as_of.isoformat()}")
scores.round(4)

### 3. 費用控除後の資産指数を見る

全系列を初期資産1で表示します。初期の未評価区間はcashで、モデルの優劣を意味しません。色だけに依存しないよう線種を変えます。

In [ ]:
figure = go.Figure()
for name, dash in zip(
    ("zero", "mean", "ridge", "tree", "buy_hold"),
    ("dot", "solid", "dash", "longdash", "dashdot"),
    strict=True,
):
    equity = comparison.backtests[name].equity
    figure.add_scatter(
        x=equity.index, y=equity, mode="lines",
        name=name, line={"dash": dash},
    )
figure.update_layout(
    title="合成DEMO:ALPHAのOOS比較 (1期遅れ、費用5bps)",
    xaxis_title="判断時刻 (UTC)",
    yaxis_title="費用控除後の資産指数 (初期=1)",
    template="plotly_white",
)
figure

## Checks

予測、評価対象リターン、費用条件を同じにした比較です。lockbox以降のラベルは採点しません。合成結果は実市場の予測性能を示しません。

In [ ]:
assert set(comparison.backtests) == {"zero", "mean", "ridge", "tree", "buy_hold"}
assert comparison.commission_bps == 3
assert comparison.slippage_bps == 2
assert comparison.evaluation.table["label_available_at"].lt(lockbox_start).all()
assert comparison.as_of < lockbox_start
assert all(
    result.net_returns.index.max() < lockbox_start
    for result in comparison.backtests.values()
)
assert comparison.backtests["zero"].equity.iloc[-1] == 1.0
print("同一費用・lag1・lockbox前の評価を確認")

## Next Steps

04では同じ合成価格を現在から見た仮想配分のリスクに使います。そのretrospective計算をPITバックテストと取り違えないでください。